In [2]:
!pip install PyFlyt stable-baselines3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 32.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.6/215.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 55.7 MB/s eta 0:00:00
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873464 sha256=bcefa31ecff2d9725fd791ed27ad00f7e28ebf174e7435f5b398515ce930eca9
  Stored in directory: /root/.cache/pip/wheels/72/95/1d/b336e5ee612ae9a019bfff4dc0bedd100ee6f0570db205fdf8
Successfully built pybullet
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency

In [1]:
import PyFlyt
import gymnasium as gym
import stable_baselines3 as sb3
from importlib.metadata import version

print("PyFlyt version:", version("PyFlyt"))
print("Gymnasium version:", gym.__version__)
print("Stable-Baselines3 version:", sb3.__version__)

PyFlyt version: 0.29.0
Gymnasium version: 1.3.0
Stable-Baselines3 version: 2.9.0


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class FlattenWaypointObs(gym.ObservationWrapper):
    """Flattens the Dict observation into a fixed 24-dim vector: [attitude(21), next_target_delta(3)]."""

    def __init__(self, env):
        super().__init__(env)
        attitude_dim = env.observation_space["attitude"].shape[0]
        flat_dim = attitude_dim + 3
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(flat_dim,), dtype=np.float64
        )

    def observation(self, obs):
        attitude = obs["attitude"]
        target_deltas = obs["target_deltas"]
        next_delta = target_deltas[0] if len(target_deltas) > 0 else np.zeros(3, dtype=np.float64)
        return np.concatenate([attitude, next_delta]).astype(np.float64)


class ClipAction(gym.ActionWrapper):
    """Clips the agent's raw action into the environment's valid action bounds."""

    def __init__(self, env):
        super().__init__(env)
        self.low = env.action_space.low
        self.high = env.action_space.high

    def action(self, action):
        return np.clip(action, self.low, self.high)

In [4]:
import PyFlyt.gym_envs
from stable_baselines3.common.monitor import Monitor


def make_env():
    """Builds the fully wrapped, SB3-compatible training environment."""
    env = gym.make("PyFlyt/QuadX-Waypoints-v4", render_mode=None)
    env = FlattenWaypointObs(env)   # dict -> 24-dim vector
    env = ClipAction(env)           # clip actions to valid bounds
    env = Monitor(env)              # SB3 episode stat logging
    return env


# Quick sanity check on the wrapped env
env = make_env()
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

obs, info = env.reset(seed=0)
print("Obs shape:", obs.shape)

Observation space: Box(-inf, inf, (24,), float64)
Action space: Box([-3.14159265 -3.14159265 -3.14159265  0.        ], [3.14159265 3.14159265 3.14159265 0.8       ], (4,), float64)
Obs shape: (24,)


In [ ]:
from stable_baselines3 import PPO

# Create the PPO model with SB3's well-tested defaults
model = PPO(
    "MlpPolicy",           # standard MLP actor-critic, matches our hand-written one
    env,
    verbose=1,             # print training logs (ep_rew_mean etc.)
    device="cpu",          # PyBullet is CPU-bound; GPU gives little benefit here
    seed=0,
)

# First training run: 200k timesteps
model.learn(total_timesteps=200_000)

[progress-bar output trimmed for repository size]

Using cpu device
Wrapping the env in a DummyVecEnv.


[progress-bar output trimmed for repository size]

| time/                   |             |
|    fps                  | 254         |
|    iterations           | 97          |
|    time_elapsed         | 782         |
|    total_timesteps      | 198656      |
| train/                  |             |
|    approx_kl            | 0.013461754 |
|    clip_fraction        | 0.131       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.55       |
|    explained_variance   | 0.82        |
|    learning_rate        | 0.0003      |
|    loss                 | 43.8        |
|    n_updates            | 960         |
|    policy_gradient_loss | -0.0192     |
|    std                  | 0.757       |
|    value_loss           | 157         |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 284         |
|    ep_rew_mean          | 189         |
| time/                  

In [ ]:
# Save the trained model FIRST — protect the 200k training result
model.save("ppo_quadx_waypoints_200k")

import os
size_mb = os.path.getsize("ppo_quadx_waypoints_200k.zip") / 1e6
print(f"Model saved: ppo_quadx_waypoints_200k.zip ({size_mb:.2f} MB)")

Model saved: ppo_quadx_waypoints_200k.zip (0.18 MB)


In [ ]:
import numpy as np
import imageio

# Build a separate env with RGB rendering (training env had render_mode=None)
render_env = gym.make("PyFlyt/QuadX-Waypoints-v4", render_mode="rgb_array")
render_env = FlattenWaypointObs(render_env)
render_env = ClipAction(render_env)
# NOTE: no Monitor here — we don't need episode logging for rendering

# Record one episode with the trained policy
obs, info = render_env.reset(seed=42)
frames = []
terminated = False
truncated = False
total_reward = 0.0

while not (terminated or truncated):
    frame = render_env.render()
    frames.append(frame[:, :, :3])  # drop alpha channel (RGBA -> RGB)

    # Use the trained model to pick actions (deterministic for clean evaluation)
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = render_env.step(action)
    total_reward += reward

# Capture the final frame
frames.append(render_env.render()[:, :, :3])

print(f"Episode finished: {len(frames)} frames")
print(f"Total reward: {total_reward:.1f}")
print(f"Targets reached: {info['num_targets_reached']}")
print(f"Out of bounds: {info['out_of_bounds']} | Collision: {info['collision']}")

# Save as video
imageio.mimsave("trained_agent.mp4", frames, fps=30)
print("Saved to trained_agent.mp4")

/usr/local/lib/python3.12/dist-packages/gymnasium/utils/passive_env_checker.py:282: UserWarning: WARN: RGB-array rendering should return a numpy array in which the last axis has three dimensions, got 4
  logger.warn(


Episode finished: 303 frames
Total reward: 316.8
Targets reached: 1
Out of bounds: False | Collision: False
Saved to trained_agent.mp4


In [ ]:
from IPython.display import Video
Video("trained_agent.mp4", embed=True, width=480)

[embedded flight video removed — see figures/ for stills]


In [ ]:
# Numerical evaluation over multiple episodes — more reliable than one video
NUM_EVAL_EPISODES = 20

eval_env = make_env()

episode_rewards = []
episode_targets = []
episode_oob = []
episode_lengths = []

for ep in range(NUM_EVAL_EPISODES):
    obs, info = eval_env.reset(seed=1000 + ep)
    terminated = truncated = False
    ep_reward = 0.0
    ep_len = 0

    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_reward += reward
        ep_len += 1

    episode_rewards.append(ep_reward)
    episode_targets.append(info["num_targets_reached"])
    episode_oob.append(info["out_of_bounds"])
    episode_lengths.append(ep_len)

import numpy as np
print(f"Episodes evaluated: {NUM_EVAL_EPISODES}")
print(f"Mean reward:        {np.mean(episode_rewards):.1f} +/- {np.std(episode_rewards):.1f}")
print(f"Mean targets:       {np.mean(episode_targets):.2f} (max possible: 4)")
print(f"Mean episode len:   {np.mean(episode_lengths):.1f}")
print(f"Out-of-bounds rate: {np.mean(episode_oob) * 100:.0f}%")
print(f"Targets per episode: {episode_targets}")

Episodes evaluated: 20
Mean reward:        228.3 +/- 150.0
Mean targets:       0.70 (max possible: 4)
Mean episode len:   278.0
Out-of-bounds rate: 0%
Targets per episode: [0, 3, 0, 0, 1, 0, 0, 0, 2, 1, 1, 0, 0, 0, 0, 3, 0, 2, 0, 1]


In [ ]:
  from stable_baselines3 import PPO

  single_env = make_env()

  model_v2 = PPO(
      "MlpPolicy",
      single_env,
      verbose=1,
      device="cpu",
      seed=0,
  )

  model_v2.learn(total_timesteps=1_000_000)

[progress-bar output trimmed for repository size]

| time/                   |            |
|    fps                  | 66         |
|    iterations           | 488        |
|    time_elapsed         | 15102      |
|    total_timesteps      | 999424     |
| train/                  |            |
|    approx_kl            | 0.06646458 |
|    clip_fraction        | 0.425      |
|    clip_range           | 0.2        |
|    entropy_loss         | 0.14       |
|    explained_variance   | 0.783      |
|    learning_rate        | 0.0003     |
|    loss                 | 142        |
|    n_updates            | 4870       |
|    policy_gradient_loss | -0.0211    |
|    std                  | 0.238      |
|    value_loss           | 371        |
----------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 233        |
|    ep_rew_mean          | 496        |
| time/                   |            |
|    f

In [ ]:
model_v2.save("ppo_quadx_waypoints_1M")

import os
size_mb = os.path.getsize("ppo_quadx_waypoints_1M.zip") / 1e6
print(f"Model saved: ppo_quadx_waypoints_1M.zip ({size_mb:.2f} MB)")

Model saved: ppo_quadx_waypoints_1M.zip (0.18 MB)


In [ ]:
from google.colab import files
files.download("ppo_quadx_waypoints_1M.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
from stable_baselines3 import PPO
model_v2 = PPO.load("ppo_quadx_waypoints_1M")
print("Model loaded.")

Model loaded.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [6]:
import numpy as np

eval_env = make_env()

NUM_EVAL_EPISODES = 20
episode_rewards, episode_targets, episode_oob, episode_lengths = [], [], [], []

for ep in range(NUM_EVAL_EPISODES):
    obs, info = eval_env.reset(seed=1000 + ep)
    terminated = truncated = False
    ep_reward, ep_len = 0.0, 0
    while not (terminated or truncated):
        action, _ = model_v2.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_reward += reward
        ep_len += 1
    episode_rewards.append(ep_reward)
    episode_targets.append(info["num_targets_reached"])
    episode_oob.append(info["out_of_bounds"])
    episode_lengths.append(ep_len)

eval_env.close()

print(f"Episodes: {NUM_EVAL_EPISODES}")
print(f"Mean reward:        {np.mean(episode_rewards):.1f} +/- {np.std(episode_rewards):.1f}")
print(f"Mean targets:       {np.mean(episode_targets):.2f} (max: 4)")
print(f"Mean episode len:   {np.mean(episode_lengths):.1f}")
print(f"Out-of-bounds rate: {np.mean(episode_oob) * 100:.0f}%")
print(f"Targets per episode: {episode_targets}")

Episodes: 20
Mean reward:        438.6 +/- 189.2
Mean targets:       3.00 (max: 4)
Mean episode len:   234.4
Out-of-bounds rate: 5%
Targets per episode: [3, 4, 4, 0, 4, 2, 3, 4, 4, 4, 4, 3, 2, 1, 3, 4, 3, 4, 3, 1]


In [7]:
import numpy as np

eval_env = make_env()

NUM_EVAL_EPISODES = 20
results = []

for ep in range(NUM_EVAL_EPISODES):
    obs, info = eval_env.reset(seed=1000 + ep)
    terminated = truncated = False
    ep_reward, ep_len = 0.0, 0
    while not (terminated or truncated):
        action, _ = model_v2.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_reward += reward
        ep_len += 1
    results.append({
        "seed": 1000 + ep,
        "targets": info["num_targets_reached"],
        "reward": ep_reward,
        "length": ep_len,
        "out_of_bounds": info["out_of_bounds"],
        "collision": info["collision"],
    })

eval_env.close()

# Print detailed breakdown, sorted by targets reached
for r in sorted(results, key=lambda x: x["targets"]):
    print(f"seed={r['seed']:5d} | targets={r['targets']} | reward={r['reward']:7.1f} | "
          f"len={r['length']:4d} | oob={r['out_of_bounds']} | collision={r['collision']}")

seed= 1003 | targets=0 | reward=  -67.0 | len=  60 | oob=False | collision=True
seed= 1013 | targets=1 | reward=   59.8 | len= 128 | oob=False | collision=True
seed= 1019 | targets=1 | reward=   58.9 | len= 126 | oob=False | collision=True
seed= 1005 | targets=2 | reward=  419.6 | len= 302 | oob=False | collision=False
seed= 1012 | targets=2 | reward=  289.6 | len= 269 | oob=True | collision=False
seed= 1000 | targets=3 | reward=  526.8 | len= 302 | oob=False | collision=False
seed= 1006 | targets=3 | reward=  492.1 | len= 302 | oob=False | collision=False
seed= 1011 | targets=3 | reward=  527.3 | len= 302 | oob=False | collision=False
seed= 1014 | targets=3 | reward=  501.6 | len= 302 | oob=False | collision=False
seed= 1016 | targets=3 | reward=  541.5 | len= 302 | oob=False | collision=False
seed= 1018 | targets=3 | reward=  509.9 | len= 302 | oob=False | collision=False
seed= 1001 | targets=4 | reward=  559.8 | len= 226 | oob=False | collision=False
seed= 1002 | targets=4 | reward=

In [8]:
import torch
import numpy as np

def get_action_std(model, obs):
    """Extracts the Gaussian policy's std (per action dim) for a given obs."""
    obs_tensor, _ = model.policy.obs_to_tensor(obs)
    with torch.no_grad():
        distribution = model.policy.get_distribution(obs_tensor)
    std = distribution.distribution.stddev.cpu().numpy()[0]  # shape (4,)
    return std

def run_episode_with_uncertainty(model, env, seed):
    obs, info = env.reset(seed=seed)
    terminated = truncated = False
    history = []
    step = 0
    while not (terminated or truncated):
        std = get_action_std(model, obs)
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        history.append({"step": step, "mean_std": std.mean(), "reward": reward})
        step += 1
    return history, info

eval_env = make_env()

# Known crash seed
crash_history, crash_info = run_episode_with_uncertainty(model_v2, eval_env, seed=1003)

# Known success seed
success_history, success_info = run_episode_with_uncertainty(model_v2, eval_env, seed=1001)

eval_env.close()

print(f"CRASH episode (seed 1003, targets={crash_info['num_targets_reached']}), last 15 steps:")
for h in crash_history[-15:]:
    print(f"  step={h['step']:3d}  mean_std={h['mean_std']:.4f}  reward={h['reward']:.2f}")

print(f"\nSUCCESS episode (seed 1001, targets={success_info['num_targets_reached']}), last 15 steps:")
for h in success_history[-15:]:
    print(f"  step={h['step']:3d}  mean_std={h['mean_std']:.4f}  reward={h['reward']:.2f}")

CRASH episode (seed 1003, targets=0), last 15 steps:
  step= 45  mean_std=0.2348  reward=0.79
  step= 46  mean_std=0.2348  reward=0.86
  step= 47  mean_std=0.2348  reward=0.87
  step= 48  mean_std=0.2348  reward=0.94
  step= 49  mean_std=0.2348  reward=0.94
  step= 50  mean_std=0.2348  reward=0.99
  step= 51  mean_std=0.2348  reward=1.01
  step= 52  mean_std=0.2348  reward=1.08
  step= 53  mean_std=0.2348  reward=1.09
  step= 54  mean_std=0.2348  reward=1.24
  step= 55  mean_std=0.2348  reward=1.27
  step= 56  mean_std=0.2348  reward=1.45
  step= 57  mean_std=0.2348  reward=1.60
  step= 58  mean_std=0.2348  reward=1.71
  step= 59  mean_std=0.2348  reward=-99.86
SUCCESS episode (seed 1001, targets=4), last 15 steps:
  step=211  mean_std=0.2348  reward=0.97
  step=212  mean_std=0.2348  reward=0.99
  step=213  mean_std=0.2348  reward=1.01
  step=214  mean_std=0.2348  reward=1.04
  step=215  mean_std=0.2348  reward=1.10
  step=216  mean_std=0.2348  reward=1.15
  step=217  mean_std=0.2348  

In [9]:
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.policies import ActorCriticPolicy


class DropoutMlpExtractor(nn.Module):
    """Same (64,64) architecture as SB3's default extractor, but with dropout
    layers inserted so MC Dropout can be used at inference time."""

    def __init__(self, feature_dim, dropout_rate=0.1):
        super().__init__()
        hidden = 64

        self.policy_net = nn.Sequential(
            nn.Linear(feature_dim, hidden), nn.Tanh(), nn.Dropout(dropout_rate),
            nn.Linear(hidden, hidden), nn.Tanh(), nn.Dropout(dropout_rate),
        )
        self.value_net = nn.Sequential(
            nn.Linear(feature_dim, hidden), nn.Tanh(), nn.Dropout(dropout_rate),
            nn.Linear(hidden, hidden), nn.Tanh(), nn.Dropout(dropout_rate),
        )

        self.latent_dim_pi = hidden
        self.latent_dim_vf = hidden

    def forward(self, features):
        return self.forward_actor(features), self.forward_critic(features)

    def forward_actor(self, features):
        return self.policy_net(features)

    def forward_critic(self, features):
        return self.value_net(features)


class DropoutActorCriticPolicy(ActorCriticPolicy):
    """ActorCriticPolicy using our dropout-enabled extractor instead of the default."""

    def _build_mlp_extractor(self) -> None:
        self.mlp_extractor = DropoutMlpExtractor(self.features_dim, dropout_rate=0.1)

In [10]:
# Build a new model with the dropout-enabled policy
temp_env = make_env()
new_model = PPO(DropoutActorCriticPolicy, temp_env, verbose=0, device="cpu", seed=0)


def copy_linear_weights(source_seq, target_seq):
    """Copies weights between two nn.Sequential modules, matching only Linear layers in order."""
    source_linears = [m for m in source_seq if isinstance(m, nn.Linear)]
    target_linears = [m for m in target_seq if isinstance(m, nn.Linear)]
    assert len(source_linears) == len(target_linears), "Layer count mismatch"
    for src, tgt in zip(source_linears, target_linears):
        tgt.load_state_dict(src.state_dict())


# Transfer the learned weights from the clean 1M model into the dropout policy
copy_linear_weights(model_v2.policy.mlp_extractor.policy_net, new_model.policy.mlp_extractor.policy_net)
copy_linear_weights(model_v2.policy.mlp_extractor.value_net, new_model.policy.mlp_extractor.value_net)
new_model.policy.action_net.load_state_dict(model_v2.policy.action_net.state_dict())
new_model.policy.value_net.load_state_dict(model_v2.policy.value_net.state_dict())
new_model.policy.log_std.data.copy_(model_v2.policy.log_std.data)

temp_env.close()
print("Weights transferred from clean 1M model into dropout-enabled policy.")

Weights transferred from clean 1M model into dropout-enabled policy.


In [11]:
test_env = make_env()
obs, info = test_env.reset(seed=1003)
test_env.close()

obs_tensor, _ = new_model.policy.obs_to_tensor(obs)

# Enable dropout at inference time (normally it's off during eval)
new_model.policy.mlp_extractor.train()

with torch.no_grad():
    actions = []
    for _ in range(5):
        dist = new_model.policy.get_distribution(obs_tensor)
        action = dist.distribution.mean  # mean action under current dropout mask
        actions.append(action.cpu().numpy()[0])

for i, a in enumerate(actions):
    print(f"Pass {i}: {a}")

Pass 0: [ 4.507501   -2.1022546   0.92421365 -0.45563146]
Pass 1: [ 3.8936064  -2.3339694   1.5888574  -0.09651639]
Pass 2: [ 4.2336187  -2.2068958   1.6870081   0.44169056]
Pass 3: [ 5.1526217  -1.5475966   1.3047118  -0.16308388]
Pass 4: [ 5.729922  -1.9996843  1.3754021 -0.5701534]


In [12]:
# Compare: raw mean action from the ORIGINAL (non-dropout) trained model, same state
obs_tensor_orig, _ = model_v2.policy.obs_to_tensor(obs)

with torch.no_grad():
    dist_orig = model_v2.policy.get_distribution(obs_tensor_orig)
    mean_action_orig = dist_orig.distribution.mean.cpu().numpy()[0]

print("Original model_v2 raw mean action:", mean_action_orig)

# Also check dropout-model with dropout DISABLED (eval mode) for a clean comparison
new_model.policy.mlp_extractor.eval()
with torch.no_grad():
    dist_new_eval = new_model.policy.get_distribution(obs_tensor)
    mean_action_new_eval = dist_new_eval.distribution.mean.cpu().numpy()[0]

print("Dropout model (dropout OFF) raw mean action:", mean_action_new_eval)

Original model_v2 raw mean action: [ 4.860271   -2.3819783   1.2747233  -0.08841026]
Dropout model (dropout OFF) raw mean action: [ 4.860271   -2.3819785   1.274723   -0.08841051]


In [15]:
# Rebuild clean: fresh dropout policy with weights from the untouched 1M model
temp_env = make_env()
new_model = PPO(DropoutActorCriticPolicy, temp_env, verbose=0, device="cpu", seed=0)

copy_linear_weights(model_v2.policy.mlp_extractor.policy_net, new_model.policy.mlp_extractor.policy_net)
copy_linear_weights(model_v2.policy.mlp_extractor.value_net, new_model.policy.mlp_extractor.value_net)
new_model.policy.action_net.load_state_dict(model_v2.policy.action_net.state_dict())
new_model.policy.value_net.load_state_dict(model_v2.policy.value_net.state_dict())
new_model.policy.log_std.data.copy_(model_v2.policy.log_std.data)

new_model.policy.mlp_extractor.eval()  # dropout OFF by default
temp_env.close()

# Verify transfer is clean
obs_tensor, _ = new_model.policy.obs_to_tensor(obs)
with torch.no_grad():
    check = new_model.policy.get_distribution(obs_tensor).distribution.mean.cpu().numpy()[0]
print("Should match original:", check)

Should match original: [ 4.860271   -2.3819785   1.274723   -0.08841051]


In [16]:
import numpy as np
import torch


def get_mc_dropout_uncertainty(model, obs, n_samples=20):
    """Runs n stochastic forward passes with dropout active, returns the
    spread of the resulting mean actions — our epistemic uncertainty signal."""
    obs_tensor, _ = model.policy.obs_to_tensor(obs)

    model.policy.mlp_extractor.train()   # enable dropout
    actions = []
    with torch.no_grad():
        for _ in range(n_samples):
            dist = model.policy.get_distribution(obs_tensor)
            actions.append(dist.distribution.mean.cpu().numpy()[0])
    model.policy.mlp_extractor.eval()    # restore deterministic mode

    actions = np.array(actions)          # (n_samples, action_dim)
    return actions.std(axis=0).mean()    # scalar uncertainty score


def run_episode_with_uncertainty(model, env, seed, n_samples=20):
    """Runs one episode using the DETERMINISTIC policy, logging MC-dropout
    uncertainty at each step (measurement only — does not affect actions)."""
    obs, info = env.reset(seed=seed)
    terminated = truncated = False
    history = []
    step = 0

    while not (terminated or truncated):
        uncertainty = get_mc_dropout_uncertainty(model, obs, n_samples)

        # Act deterministically (dropout is off here — same policy as the 1M model)
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)

        history.append({"step": step, "uncertainty": uncertainty, "reward": reward})
        step += 1

    return history, info


eval_env = make_env()

crash_hist, crash_info = run_episode_with_uncertainty(new_model, eval_env, seed=1003)
success_hist, success_info = run_episode_with_uncertainty(new_model, eval_env, seed=1001)

eval_env.close()

print(f"CRASH (seed 1003, targets={crash_info['num_targets_reached']}, "
      f"collision={crash_info['collision']}), last 15 steps:")
for h in crash_hist[-15:]:
    print(f"  step={h['step']:3d}  uncertainty={h['uncertainty']:.4f}  reward={h['reward']:7.2f}")

print(f"\nSUCCESS (seed 1001, targets={success_info['num_targets_reached']}), last 15 steps:")
for h in success_hist[-15:]:
    print(f"  step={h['step']:3d}  uncertainty={h['uncertainty']:.4f}  reward={h['reward']:7.2f}")

print(f"\nCRASH mean uncertainty:   {np.mean([h['uncertainty'] for h in crash_hist]):.4f}")
print(f"SUCCESS mean uncertainty: {np.mean([h['uncertainty'] for h in success_hist]):.4f}")

CRASH (seed 1003, targets=0, collision=True), last 15 steps:
  step= 45  uncertainty=0.4987  reward=   0.79
  step= 46  uncertainty=0.5096  reward=   0.86
  step= 47  uncertainty=0.4629  reward=   0.87
  step= 48  uncertainty=0.5085  reward=   0.94
  step= 49  uncertainty=0.4982  reward=   0.94
  step= 50  uncertainty=0.4450  reward=   0.99
  step= 51  uncertainty=0.4348  reward=   1.01
  step= 52  uncertainty=0.5563  reward=   1.08
  step= 53  uncertainty=0.5015  reward=   1.09
  step= 54  uncertainty=0.4446  reward=   1.24
  step= 55  uncertainty=0.5355  reward=   1.27
  step= 56  uncertainty=0.6165  reward=   1.45
  step= 57  uncertainty=0.4498  reward=   1.60
  step= 58  uncertainty=0.4765  reward=   1.71
  step= 59  uncertainty=0.5149  reward= -99.86
SUCCESS (seed 1001, targets=3), last 15 steps:
  step=287  uncertainty=0.4989  reward=   1.57
  step=288  uncertainty=0.4804  reward=   1.62
  step=289  uncertainty=0.5282  reward=   1.66
  step=290  uncertainty=0.4561  reward=   1.70

In [17]:
import numpy as np

eval_env = make_env()
episode_summaries = []

for ep in range(20):
    seed = 1000 + ep
    hist, info = run_episode_with_uncertainty(new_model, eval_env, seed, n_samples=15)
    episode_summaries.append({
        "seed": seed,
        "targets": info["num_targets_reached"],
        "collision": info["collision"],
        "oob": info["out_of_bounds"],
        "mean_uncertainty": np.mean([h["uncertainty"] for h in hist]),
        "max_uncertainty": np.max([h["uncertainty"] for h in hist]),
        "last5_uncertainty": np.mean([h["uncertainty"] for h in hist[-5:]]),
    })

eval_env.close()

for s in sorted(episode_summaries, key=lambda x: x["targets"]):
    print(f"seed={s['seed']:5d} | targets={s['targets']} | collision={s['collision']} | "
          f"mean_unc={s['mean_uncertainty']:.4f} | max_unc={s['max_uncertainty']:.4f} | "
          f"last5_unc={s['last5_uncertainty']:.4f}")

seed= 1003 | targets=0 | collision=True | mean_unc=0.5077 | max_unc=0.6700 | last5_unc=0.4732
seed= 1019 | targets=1 | collision=True | mean_unc=0.4926 | max_unc=0.6709 | last5_unc=0.4697
seed= 1005 | targets=2 | collision=False | mean_unc=0.4856 | max_unc=0.6497 | last5_unc=0.4868
seed= 1011 | targets=2 | collision=False | mean_unc=0.4933 | max_unc=0.6585 | last5_unc=0.4674
seed= 1012 | targets=2 | collision=False | mean_unc=0.4925 | max_unc=0.6579 | last5_unc=0.4820
seed= 1013 | targets=2 | collision=False | mean_unc=0.4929 | max_unc=0.6934 | last5_unc=0.4832
seed= 1000 | targets=3 | collision=True | mean_unc=0.4963 | max_unc=0.6760 | last5_unc=0.4950
seed= 1001 | targets=3 | collision=False | mean_unc=0.4702 | max_unc=0.7382 | last5_unc=0.5290
seed= 1016 | targets=3 | collision=False | mean_unc=0.4600 | max_unc=0.6485 | last5_unc=0.4115
seed= 1018 | targets=3 | collision=False | mean_unc=0.4835 | max_unc=0.6443 | last5_unc=0.4982
seed= 1002 | targets=4 | collision=False | mean_unc=0